<a href="https://colab.research.google.com/github/sohail574108/lab-extractive-question-answering/blob/main/lab_extractive_question_answering(6)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB | Extractive Question Answering

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

Install Dependencies

In [ ]:
!pip install -U langchain langchain-core langchain-classic langchain-pinecone langchain-huggingface datasets pinecone-client sentence-transformers torch

In [ ]:
import getpass
import os

from pinecone import Pinecone, ServerlessSpec

# Clear PINECONE_API_KEY from environment to ensure re-prompting if needed
if "PINECONE_API_KEY" in os.environ:
    del os.environ["PINECONE_API_KEY"]

# Prompt for API key if not set
if not os.getenv("PINECONE_API_KEY"):
    api_key_input = getpass.getpass("Enter your Pinecone API key: ")
    os.environ["PINECONE_API_KEY"] = api_key_input

pinecone_api_key = os.environ.get("PINECONE_API_KEY")

# Verify the key is set (print masked version)
if pinecone_api_key:
    print(f"Pinecone API Key (masked): {pinecone_api_key[:4]}...{pinecone_api_key[-4:]}")
else:
    print("Pinecone API Key is not set.")

Enter your Pinecone API key: ··········
Pinecone API Key (masked): pcsk...5VoN


# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [ ]:
from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("squad", split="train").to_pandas()

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:

    # select only title and context column
df = df[["title", "context"]]
    # drop rows containing duplicate context passages
df = df.drop_duplicates(subset=["context"])
    # reset the index
df = df.reset_index(drop=True)
df

,title,context
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,The university is the major seat of the Congre...
3,University_of_Notre_Dame,The College of Engineering was established in ...
4,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...
...,...,...
18886,Kathmandu,"Institute of Medicine, the central college of ..."
18887,Kathmandu,Football and Cricket are the most popular spor...
18888,Kathmandu,The total length of roads in Nepal is recorded...
18889,Kathmandu,The main international airport serving Kathman...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our context passages which we can retrieve using another vector (query vector). We first need to initialize our connection to Pinecone to create our vector index. For this, we need a free [API key]("https://app.pinecone.io/"), and then we initialize the connection like so:

In [ ]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key, environment=spec.region)

Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [ ]:
index_name = "question-answer"


# Check if index exists & create with ServerlessSpec
if index_name not in pc.list_indexes().names():
    #create the index if it does not exist
    pc.create_index(index_name, dimension=384, metric="cosine", spec=spec)

# Connect to index
index = pc.Index(index_name)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [ ]:
!pip install sentence-transformers torch langchain-huggingface

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# Set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load retriever model (multi-qa-MiniLM-L6-cos-v1 - 384 dimensions)
retriever = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1") #use the 'multi-qa-MiniLM-L6-cos-v1' model from HuggingFace to build the retriever

# ✅ Now retriever is loaded and ready
print("Retriever model loaded:", retriever)

Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever model loaded: SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [ ]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:i_end]
    # generate embeddings for batch
    emb = retriever.encode(batch["context"].tolist()).tolist()
    # get metadata
    meta = [{'text': text, 'title': title} for text, title in zip(batch['context'], batch['title'])]
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb, meta))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/296 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 18891}},
 'total_vector_count': 18891,
 'vector_type': 'dense'}

# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [ ]:
!pip install transformers==4.29.1 && pip install huggingface-hub<1.0.0,>=0.33.4 --force-reinstall
import torch
from transformers import pipeline

# Set device to GPU if available (moved from earlier cell for robustness)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device for reader: {device}")

model_name = 'deepset/electra-base-squad2'
# load the reader model into a question-answering pipeline
reader = pipeline(tokenizer=model_name, model=model_name, task='question-answering', device=device)
reader

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 27.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 65.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
Using device for reader: cuda


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda


Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [ ]:
# gets context passages from the pinecone index
def get_context(question, top_k):
    # generate embeddings for the question
    xq = retriever.encode(question).tolist()
    # search pinecone index for context passage with the answer
    xc = index.query(
        vector=xq,
        top_k=top_k,
        include_metadata=True
    )
    # extract the context passage from pinecone search result
    c = [match["metadata"]["text"] for match in xc["matches"]]
    return c



In [ ]:
from pprint import pprint

# extracts answer from the context passage
def extract_answer(question, context):
    results = []
    for c in context:
        # feed the reader the question and contexts to extract answers
        answer = reader(question=question, context=c)
        # add the context to answer dict for printing both together
        answer["context"] = c
        results.append(answer)
    # sort the result based on the score from reader model
    sorted_result = pprint(sorted(results, key=lambda x: x['score'], reverse=True))
    return sorted_result


In [ ]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 3)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.',
 'Egypt has a developed energy market based on coal, oil, natural gas, and hydro power. Substantial coal deposits in the northeast Sinai are mined at the rate of about 600,000 tonnes (590,000 long tons; 660,000 short tons) per year. Oil and gas are produced in the western desert regions, the Gulf of Suez, and the Nile Delta. Egypt has huge reserves of gas,

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [ ]:
extract_answer(question, context)

[{'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of '
             'natural gas (in 2013), which makes Egypt as the largest oil '
             'producer not member of the Organization of the Petroleum '
             'Exporting Countries (OPEC) and the second-largest dry natural '
             'gas producer in Africa. In 2013, Egypt was the largest consumer '
             'of oil and natural gas in Africa, as more than 20% of total oil '
             'consumption and more than 40% of total dry natural gas '
             'consumption in Africa. Also, Egypt possesses the largest oil '
             'refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is '
             'currently planning to build its first nuclear power plant in El '
             'Dabaa city, northern Egypt.',
  'end': 33,
  'score': 0.9999855201993455,
  'start': 20},
 {'answer': '600,000 tonnes',
  'context': 'Egypt has a developed energy market based on coal, oi

The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [ ]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, '
             'Hurley and Chen developed the idea for YouTube during the early '
             'months of 2005, after they had experienced difficulty sharing '
             "videos that had been shot at a dinner party at Chen's apartment "
             'in San Francisco. Karim did not attend the party and denied that '
             'it had occurred, but Chen commented that the idea that YouTube '
             'was founded after a dinner party "was probably very strengthened '
             'by marketing ideas around creating a story that was very '
             'digestible".',
  'end': 79,
  'score': 0.9999276399612427,
  'start': 64}]


In [ ]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity '
             'and general relativity. He also made important contributions to '
             'statistical mechanics, especially his mathematical treatment of '
             'Brownian motion, his resolution of the paradox of specific '
             'heats, and his connection of fluctuations and dissipation. '
             'Despite his reservations about its interpretation, Einstein also '
             'made contributions to quantum mechanics and, indirectly, quantum '
             'field theory, primarily through his theoretical studies of the '
             'photon.',
  'end': 86,
  'score': 0.9500371217727661,
  'start': 29}]


Let's run another question. This time for top 3 context passages from the retriever.

In [ ]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Date': 'Tue, 24 Feb 2026 15:56:30 GMT', 'Content-Type': 'text/plain', 'Content-Length': '12', 'Connection': 'keep-alive', 'x-pinecone-auth-rejected-reason': 'Malformed domain', 'www-authenticate': 'Malformed domain', 'server': 'envoy'})
HTTP response body: Unauthorized


The result looks pretty good.

In [ ]:
pc.delete_index(index_name)

### Add a few more questions. What did you observe?

In [ ]:
question = "Who was the first president of usa"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'James Buchanan in 1856',
  'context': 'Eisenhower was the last president born in the 19th century, and '
             'at age 62, was the oldest man elected President since James '
             'Buchanan in 1856 (President Truman stood at 64 in 1948 as the '
             'incumbent president at the time of his election four years '
             'earlier). Eisenhower was the only general to serve as President '
             'in the 20th century and the most recent President to have never '
             'held elected office prior to the Presidency (The other '
             'Presidents who did not have prior elected office were Zachary '
             'Taylor, Ulysses S. Grant, William Howard Taft and Herbert '
             'Hoover).',
  'end': 140,
  'score': 0.9689022898674011,
  'start': 118},
 {'answer': 'George Washington, was inaugurated',
  'context': 'In 1785, the assembly of the Congress of the Confederation made '
             'New York the national capital shortly a

In [ ]:
question = "Who was the first president of pakistan"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'President Pervez Musharraf',
  'context': '\n'
             ' Pakistan: The Olympic torch reached Islamabad for the first '
             'time ever on April 16. President Pervez Musharraf and Prime '
             'Minister Yousaf Raza Gillani spoke at the opening ceremony of '
             'the relay. Security was high, for what one newspaper called the '
             '"most sensitive leg" of the torch\'s Olympic journey. The relay '
             'was initially supposed to carry the torch around Islamabad, but '
             'the entire relay was cancelled due to security concerns '
             'regarding "militant threats or anti-China protests", and '
             'replaced by an indoors ceremony with the torch carried around '
             'the track of Jinnah Stadium. In fear of violent protests and '
             'bomb attacks, the torch relay in Pakistan took place in a '
             'stadium behind closed doors. Although the relay was behind '
             'closed

In [ ]:
question = "Who won 1992 cricket world cup"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'Dwayne Leverock',
  'context': "Bermuda's national cricket team participated in the Cricket "
             'World Cup 2007 in the West Indies. Their most famous player is a '
             '130 kilograms (290 lb) police officer named Dwayne Leverock. But '
             'India defeated Bermuda and set a record of 413 runs in a One-Day '
             'International (ODI). Bermuda were knocked out of the World Cup. '
             'Also very well-known is David Hemp, a former captain of '
             'Glamorgan in English first class cricket. The annual "Cup Match" '
             "cricket tournament between rival parishes St George's in the "
             'east and Somerset in the west is the occasion for a popular '
             'national holiday. This tournament began in 1872 when Captain '
             'Moresby of the Royal Navy introduced the game to Bermuda, '
             'holding a match at Somerset to mark forty years since the unjust '
             'thraldom of slaver

i have observed that the answers are not correct,